# Google ADK Practice
## Section 1: Environment Setup

Before we can build our agents, we need to install the necessary packages and configure our environment. Our agents will be powered by an **Azure OpenAI** deployment, connected through ADK's LiteLLM model connector.

### 1.1: Install Google ADK

First, let's install the Agent Development Kit (with the `mcp` extra used in Section 3), **LiteLLM** (ADK's connector for non-Gemini models such as Azure OpenAI), and `python-dotenv`.

**Note:** Recent LiteLLM versions require the MCP SDK 2.x (`mcp>=2.2`) when agents use tools. CrewAI currently pins `mcp` 1.x, so run this notebook and the CrewAI notebook in separate environments/kernels.

In [1]:
%pip install -q "google-adk[mcp]>=2.12" "litellm>=1.84" "mcp>=2.2" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.15.44 requires mcp<2,>=1.30.0, but you have mcp 2.3.0 which is incompatible.


### 1.2: Configure Azure OpenAI Credentials

Store your Azure OpenAI credentials in a `.env` file (in this folder or any parent folder). If a value is missing, this cell securely prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [2]:
# Day 14: direct OpenAI API setup (NOT Azure OpenAI)
import os
from pathlib import Path
from dotenv import load_dotenv

# Use the Day 8 OPENAI_API_KEY copied into day 14/.env
project_root = Path.home() / "Documents" / "GitHub" / "Bootcamp_Accenture_JB"
env_file = project_root / "day 14" / ".env"
if not env_file.is_file():
    raise FileNotFoundError(f"Missing file: {env_file}")
load_dotenv(env_file, override=True)

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key or api_key.lower() in {"api-key-here", "your-api-key", "your-openai-api-key", "sk-your-key"}:
    raise ValueError("No usable OPENAI_API_KEY in day 14/.env")

# Compatibility variables used by the existing exercise cells.
# These point to the regular OpenAI API, NOT to Azure.
AZURE_OPENAI_ENDPOINT = "https://api.openai.com/v1/"
AZURE_OPENAI_DEPLOYMENT = "gpt-4o-mini"
os.environ["AZURE_OPENAI_API_KEY"] = api_key
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_DEPLOYMENT"] = AZURE_OPENAI_DEPLOYMENT
print("Provider: OpenAI API (not Azure). Model:", AZURE_OPENAI_DEPLOYMENT)


Provider: OpenAI API (not Azure). Model: gpt-4o-mini


### 1.3: Import ADK Components

Now let's import the core building blocks we'll need:
- **LlmAgent** - The agent that orchestrates everything
- **LiteLlm** - Model connector that lets ADK use non-Gemini models such as Azure OpenAI (the brain of the agent)
- **FunctionTool** - Wraps Python functions as tools
- **InMemoryRunner** - Executes the agent
- **MCP Components** - For connecting to external services

In [3]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools.function_tool import FunctionTool

# MCP imports for Section 3
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

print("✅ All ADK components imported successfully!")

✅ All ADK components imported successfully!


### 1.4: Configure the Model and Retry Options

ADK connects to Azure OpenAI through `LiteLlm`. Azure OpenAI's v1 API is OpenAI-compatible, so we use the `openai/<deployment>` model string with `api_base` pointing at the Azure endpoint.

When working with LLMs, network issues or rate limits can occur. `num_retries` tells LiteLLM to automatically retry failed calls (for example, rate limits or transient server errors).

In [9]:
import os
from google.adk.models.lite_llm import LiteLlm

os.environ["OLLAMA_API_BASE"] = "http://127.0.0.1:11434"

azure_model = LiteLlm(
    model="ollama_chat/qwen3.5:2b-q4_K_M",
    api_base="http://127.0.0.1:11434",
    num_retries=0,
)

print("Google ADK configured for local Ollama")

Google ADK configured for local Ollama


---
## Section 2: Building Your First Agent with Custom Tools

Let's create a **Recipe Finder Agent** that can search for recipes based on ingredients you have at home.

### The Agent Creation Flow:
```
1. Define a Python function (your custom tool)
2. Wrap it with FunctionTool
3. Create an LlmAgent with the tool
4. Create a Runner to execute the agent
5. Test with queries!
```

### 2.1: Define Your Custom Tool Function

First, we create a Python function that will act as a tool. The function should:
- Have clear parameters with type hints
- Include a descriptive docstring (the agent reads this!)
- Return structured data (dict, list, or string)

**Important:** The agent uses your docstring to understand WHEN and HOW to call this function!

In [5]:
def find_recipe(ingredients: str, cuisine: str = "any") -> dict:
    """Finds a recipe based on available ingredients and optional cuisine preference.

    Args:
        ingredients: Comma-separated list of ingredients available (e.g., "chicken, tomatoes, garlic")
        cuisine: Preferred cuisine type (e.g., "Italian", "Mexican", "Asian") or "any"

    Returns:
        Dictionary with recipe details including name, ingredients needed, and instructions
    """

    # Simulated recipe database (in production, this would be a real API call)
    recipes = {
        "chicken": {
            "name": "Garlic Chicken Pasta",
            "cuisine": "Italian",
            "ingredients": ["chicken", "pasta", "garlic", "olive oil", "parmesan"],
            "instructions": "1. Cook pasta. 2. Sauté garlic and chicken. 3. Combine with pasta and cheese.",
            "prep_time": "25 minutes",
            "servings": 4
        },
        "tomato": {
            "name": "Fresh Tomato Salsa",
            "cuisine": "Mexican",
            "ingredients": ["tomatoes", "onion", "cilantro", "lime", "jalapeño"],
            "instructions": "1. Dice tomatoes and onion. 2. Mix with chopped cilantro. 3. Add lime juice.",
            "prep_time": "10 minutes",
            "servings": 6
        },
        "egg": {
            "name": "Classic Omelet",
            "cuisine": "French",
            "ingredients": ["eggs", "butter", "cheese", "herbs"],
            "instructions": "1. Beat eggs. 2. Melt butter in pan. 3. Pour eggs and fold with fillings.",
            "prep_time": "8 minutes",
            "servings": 2
        }
    }

    # Simple ingredient matching logic
    ingredient_list = [ing.strip().lower() for ing in ingredients.split(",")]

    for key_ingredient in ingredient_list:
        for recipe_key, recipe in recipes.items():
            if recipe_key in key_ingredient:
                # Check cuisine preference
                if cuisine.lower() != "any" and cuisine.lower() not in recipe["cuisine"].lower():
                    continue
                return {
                    "status": "found",
                    "recipe": recipe
                }

    return {
        "status": "not_found",
        "message": f"No recipes found for ingredients: {ingredients}"
    }

print("✅ Recipe finder function created!")

✅ Recipe finder function created!


### 2.2: Create the Agent with Your Tool

Now we'll create an agent and give it access to our `find_recipe` tool.

**Key Components:**
- `model`: The LLM that powers the agent (our Azure OpenAI deployment via `LiteLlm`)
- `name`: Identifier for your agent
- `instruction`: The agent's system prompt - tells it HOW to behave
- `tools`: List of tools the agent can use (wrapped with `FunctionTool`)

In [10]:
recipe_agent = LlmAgent(
    model=azure_model,
    name="recipe_finder_agent",
    instruction="""You are a helpful cooking assistant that helps users find recipes.

    When a user asks for recipe suggestions:
    1. Use the find_recipe tool with the ingredients they mention
    2. Present the recipe in a friendly, easy-to-read format
    3. Highlight the prep time and number of servings
    4. If no recipe is found, suggest alternative ingredients or cuisines

    Keep your responses warm and encouraging!""",
    tools=[FunctionTool(func=find_recipe)]
)

print("✅ Recipe Finder Agent created!")

✅ Recipe Finder Agent created!


### 2.3: Create a Runner to Execute the Agent

The **Runner** is responsible for executing your agent. Think of it as the engine that runs the agent.

We're using `InMemoryRunner` - the simplest runner type, perfect for testing and development.

In [11]:
recipe_runner = InMemoryRunner(agent=recipe_agent)

print("✅ Runner initialized!")

✅ Runner initialized!


### 2.4: Test Your Agent!

Now let's test our agent by asking it to find recipes.

**Notice:** We use `await` because ADK operations are asynchronous.

In [12]:
# Test 1: Simple ingredient search
response = await recipe_runner.run_debug("I have chicken and garlic. What can I make?")

# Extract and print the response
for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)

recipe_finder_agent > That sounds perfect! Based on your ingredients of chicken and garlic, here's a delicious suggestion for you:

**🍝 Garlic Chicken Pasta - Italian Recipe**

This is a classic simple pasta dish that uses just the two main ingredients you have (plus a few others).

**⏱️ Time to make:** 25 minutes
**🍽️ Servings:** 4 people

**What you'll need:**
- Chicken
- Pasta
- Garlic
- Olive oil
- Parmesan cheese

**Instructions:**
1. Cook your pasta according to the package instructions
2. Sauté garlic and chicken in a pan
3. Combine everything with pasta and parmesan for an authentic Italian meal

This is a great quick weeknight dinner that's perfect with just those two ingredients! Let me know if you'd like to try it out or need any more cooking tips.
The user has chicken and garlic as ingredients and wants recipe suggestions. I should use the find_recipe tool with these ingredients to help them find suitable recipes.
Great! I found a recipe that matches the user's ingredients 

In [13]:
# Test 2: Cuisine preference
response = await recipe_runner.run_debug("I want to make something Mexican with tomatoes")

for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)

recipe_finder_agent > That's a great suggestion! 🌶️ Based on having tomatoes, here's a perfect Mexican dish for you:

**🍅 Fresh Tomato Salsa - Classic Mexican Recipe**

Perfect when you want to cook with tomatoes as your main ingredient!

**⏱️ Time to make:** 10 minutes
**🍽️ Servings:** 6 people

**What you'll need:**
- Tomatoes
- Onion
- Cilantro
- Lime
- Jalapeño (for heat!)

**Instructions:**
1. Dice your tomatoes and onion into small pieces
2. Mix them with chopped fresh cilantro
3. Add lime juice for a zesty kick

This is an incredibly quick salsa you can use right away! You could pair it with tortillas, rice, or as a base for burritos and tacos. It's a vibrant, flavorful dish that really highlights the bright taste of tomatoes. Enjoy your cooking!
The user wants to make something Mexican with tomatoes, so I should use the find_recipe tool with "tomatoes" as a key ingredient. Let me check what other ingredients might be needed for a Mexican dish - typically Mexican cooking often u

---
## 🌐 Section 3: Model Context Protocol (MCP) Integration

Now let's level up! Instead of writing all tools ourselves, we can connect to **MCP Servers** - pre-built services that provide specialized tools.

### What is MCP?

**Model Context Protocol** is an open standard that connects agents to external services:

```
Your Agent <--MCP--> External Servers (GitHub, Slack, etc.)
```

**Benefits:**
- ✅ No need to write API integration code
- ✅ Community-maintained tools
- ✅ Standardized interface
- ✅ Easy to swap services

### 3.1: Adding Image Generation with MCP

Let's enhance our Recipe Assistant with the ability to generate food images using the **Everything MCP Server**.

**MCP Setup Flow:**
```
1. Choose an MCP server
2. Configure connection parameters
3. Create McpToolset
4. Add to agent's tools
```

### 3.2: Configure the MCP Toolset

We'll connect to the **Everything MCP Server** which provides a `get-tiny-image` tool.

**Connection Details:**
- Uses `npx` to run the MCP server (requires **Node.js 24+**; the MCP cells are skipped if `npx` is not found)
- Server package: `@modelcontextprotocol/server-everything`
- We filter to only use the `get-tiny-image` tool

In [15]:
import shutil
import sys

NPX_AVAILABLE = shutil.which("npx") is not None

mcp_image_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=[
                "-y",
                "@modelcontextprotocol/server-everything",
            ],
        ),
        timeout=120,  # the first run downloads the server package via npx
    ),
    tool_filter=["get-tiny-image"],
    # Jupyter's sys.stderr has no OS file handle on Windows, so give the server process the real one
    errlog=sys.__stderr__,
)

if NPX_AVAILABLE:
    print("✅ MCP Toolset configured!")
else:
    print("⚠️ MCP Toolset configured, but npx was not found. Install Node.js 24+ to run the MCP cells.")

⚠️ MCP Toolset configured, but npx was not found. Install Node.js 24+ to run the MCP cells.


### 3.3: Create Enhanced Agent with Both Tools

Now let's create an enhanced agent that has BOTH our custom `find_recipe` tool AND the MCP image generation tool.

In [16]:
enhanced_recipe_agent = LlmAgent(
    model=azure_model,
    name="enhanced_recipe_agent",
    instruction="""You are an advanced cooking assistant with image generation capabilities.

    When a user asks for recipes:
    1. Use find_recipe to search for recipes
    2. Present the recipe clearly
    3. If the user wants visual inspiration, use get-tiny-image to generate a sample image

    Always be helpful and creative!""",
    tools=[
        FunctionTool(func=find_recipe),
        mcp_image_toolset
    ]
)

enhanced_runner = InMemoryRunner(agent=enhanced_recipe_agent)

print("✅ Enhanced agent with MCP tool created!")

✅ Enhanced agent with MCP tool created!


### 3.4: Test the Enhanced Agent

Let's ask for a recipe with an image! ADK starts the MCP server (via `npx`) the first time the agent needs its tools.

In [17]:
if NPX_AVAILABLE:
    response = await enhanced_runner.run_debug(
        "I have eggs. Can you suggest a recipe and show me a sample image?"
    )

    # Display text responses
    for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)
else:
    print("Skipped: the MCP server needs Node.js 24+ (npx).")

Skipped: the MCP server needs Node.js 24+ (npx).


### 3.5: Display Generated Images

The MCP server returns base64-encoded images. Let's decode and display them!

In [18]:
from IPython.display import display, Image as IPImage
import base64

if NPX_AVAILABLE:
    # Extract and display any images from the response
    for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_response") and part.function_response:
                    for item in part.function_response.response.get("content", []):
                        if item.get("type") == "image":
                            print("\n🖼️ Generated Image:")
                            display(IPImage(data=base64.b64decode(item["data"])))
else:
    print("Skipped: the MCP server needs Node.js 24+ (npx).")

Skipped: the MCP server needs Node.js 24+ (npx).


---
## Section 4: Observability - Debugging Your Agents

When building agents, you need to understand what's happening under the hood:
- Which tools are being called?
- What are the tool inputs and outputs?
- How is the agent reasoning?

ADK provides powerful observability features to help you debug and monitor your agents.

### 4.1: Verbose Mode

The simplest way to see what's happening is to use `verbose=True` in `run_debug()`.

This prints:
- 📝 User messages
- 🤖 Agent reasoning
- 🛠️ Tool calls with parameters
- 📊 Tool responses

In [19]:
print("=" * 60)
print("🔍 VERBOSE MODE DEMONSTRATION")
print("=" * 60)

response = await recipe_runner.run_debug(
    "Find me an Italian recipe with chicken",
    verbose=True  # This enables detailed logging
)

🔍 VERBOSE MODE DEMONSTRATION
recipe_finder_agent > The user is asking for an Italian recipe with chicken. I should use the find_recipe tool with "chicken" as the ingredient and "Italian" as the cuisine preference. Let me make that call.
recipe_finder_agent > [Calling tool: find_recipe({'ingredients': 'chicken, italian'})]
recipe_finder_agent > [Tool result: {'status': 'found', 'recipe': {'name': 'Garlic Chicken Pasta', 'cuisine': 'Italian', 'ingredients': ...]
recipe_finder_agent > Great! I found the Garlic Chicken Pasta recipe for them. This is the same Italian recipe they already saw from before since it matches what was previously found. Let me present this to the user again with a nice warm tone.**🍝 Garlic Chicken Pasta - Classic Italian Recipe**

Perfect for you if you'd like an Italian dish using chicken! Here's exactly what you need:

**⏱️ Time to make:** 25 minutes
**🍽️ Servings:** 4 people

**What you'll need:**
- Chicken
- Pasta
- Garlic
- Olive oil
- Parmesan cheese

**Instr

### 4.2: Event Streaming and Inspection

For more control, you can inspect individual events. Each event contains:
- `event.content` - The message content
- `event.content.parts` - Parts of the message (text, function calls, etc.)
- Metadata like timestamps and invocation IDs

In [20]:
def analyze_events(events):
    """Helper function to analyze and categorize events."""

    text_responses = []
    tool_calls = []
    tool_responses = []

    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                # Text responses from agent
                if part.text:
                    text_responses.append(part.text)

                # Tool calls made by agent
                if hasattr(part, 'function_call') and part.function_call:
                    tool_calls.append({
                        'name': part.function_call.name,
                        'args': part.function_call.args
                    })

                # Tool responses
                if hasattr(part, 'function_response') and part.function_response:
                    tool_responses.append({
                        'name': part.function_response.name,
                        'response': part.function_response.response
                    })

    return text_responses, tool_calls, tool_responses

print("✅ Event analyzer function created!")

✅ Event analyzer function created!


### 4.3: Test Event Analysis

Let's run a query and analyze all the events that occur.

In [21]:
response = await recipe_runner.run_debug("What can I cook with tomatoes?")

text_responses, tool_calls, tool_responses = analyze_events(response)

print("\n" + "=" * 60)
print("📊 EVENT ANALYSIS")
print("=" * 60)

print(f"\n🛠️ TOOL CALLS ({len(tool_calls)}):")
for i, call in enumerate(tool_calls, 1):
    print(f"  {i}. {call['name']}")
    print(f"     Args: {call['args']}")

print(f"\n📊 TOOL RESPONSES ({len(tool_responses)}):")
for i, resp in enumerate(tool_responses, 1):
    print(f"  {i}. {resp['name']}")
    print(f"     Response: {resp['response']}")

print(f"\n💬 AGENT TEXT RESPONSES ({len(text_responses)}):")
for i, text in enumerate(text_responses, 1):
    print(f"  {i}. {text[:100]}...")  # First 100 chars

print("\n" + "=" * 60)

recipe_finder_agent > I found a few excellent tomato-based recipes for you! 🍅 Since tomatoes are so versatile, I have some great suggestions:

**1. Fresh Tomato Salsa (Mexican Cuisine)**
   ⏱️ Time to make: 10 minutes | 🍽️ Servings: 6 people

   **What you'll need:** Tomatoes, onion, cilantro, lime, jalapeño

   This is a quick, vibrant salsa perfect with tortillas, rice, or burritos! It highlights the fresh taste of tomatoes in an authentic Mexican way.

**2. Garlic Chicken Pasta (Italian Cuisine)**
   ⏱️ Time to make: 25 minutes | 🍽️ Servings: 4 people
   
   **What you'll need:** Chicken, pasta, garlic, olive oil, parmesan
   
   This is a classic Italian meal using both chicken and tomatoes (often in the traditional Parmigiano-Reggiano) for an authentic home-style pasta dish.

**3. Tomato Basil Cream Soup (Italian)**
   ⏱️ Time to make: 30 minutes | 🍽️ Servings: 4 people
   
   **What you'll need:** Potatoes, tomatoes, basil, cream

   This comforting tomato soup with fresh basil i

### 4.4: Custom Logging for Production

In production, you might want to log events to a file or monitoring service. Here's a pattern you can use:

In [22]:
import json
from datetime import datetime

def log_agent_execution(query: str, events):
    """Log agent execution for monitoring and debugging."""

    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "events": []
    }

    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    log_entry["events"].append({
                        "type": "text",
                        "content": part.text
                    })
                if hasattr(part, 'function_call') and part.function_call:
                    log_entry["events"].append({
                        "type": "tool_call",
                        "tool": part.function_call.name,
                        "args": dict(part.function_call.args)
                    })

    # In production, you'd write to a logging service
    print("\n📝 LOG ENTRY:")
    print(json.dumps(log_entry, indent=2))

    return log_entry

# Test it
response = await recipe_runner.run_debug("Quick egg recipe")
log = log_agent_execution("Quick egg recipe", response)

recipe_finder_agent > I'd love to help you find the perfect quick egg recipe! 🥚 Eggs are so versatile - there are so many delicious ways to cook them that you can choose based on your preferences and what you already have in your pantry or fridge.

Here are a few quick options I can suggest:

**1. Scrambled Egg Frittata (Quick 15-min version)**
   ⏱️ Time to make: 15 minutes | 🍽️ Servings: 4-6 people
   
   A light, fluffy scramble that pairs well with spinach or cheese! If you already have eggs in your kitchen, a quick scrambled breakfast is probably what you're thinking of.

**2. Italian Hoppin' Egg omelet (Quick 10-min version)**
   ⏱️ Time to make: 10 minutes | 🍽️ Servings: 2-3 people
   
   A fluffy, golden-brown egg omelet that's perfect for your eggs! With tomatoes already mentioned in our conversation, this could work as a great meal with fresh ingredients.

**3. Garlic Parmesan Egg Frittata (Quick 20-min version)**
   ⏱️ Time to make: 20 minutes | 🍽️ Servings: 4 people
   
   

---
## 🎯 Section 5: Final Challenge - Build Your Own Agent!

Now it's your turn to apply everything you've learned!

## 🍕 The Challenge: Nutrition Calculator Agent

Build an agent that helps users track nutritional information for their meals.

### Requirements:

1. **Create a custom tool:** `calculate_nutrition(food_item: str, serving_size: float) -> dict`
   - Should return calories, protein, carbs, and fat for common foods
   - Create a simple database with at least 5 food items
   - Example foods: rice, chicken breast, broccoli, apple, salmon

2. **Create the agent:**
   - Use the Azure OpenAI model (`azure_model`)
   - Give it clear instructions on how to help users
   - Add the nutrition calculator tool

3. **Test your agent:**
   - Ask it to calculate nutrition for a single food
   - Ask it to calculate total nutrition for a meal with multiple items
   - Use verbose mode to see the tool calls

4. **BONUS:** Add the MCP image tool so users can visualize their meals!

### Starter Code Structure:

```python
# Step 1: Define your tool
def calculate_nutrition(food_item: str, serving_size: float = 1.0) -> dict:
    # Your code here
    pass

# Step 2: Create the agent
nutrition_agent = LlmAgent(
    # Your configuration here
)

# Step 3: Create runner
nutrition_runner = InMemoryRunner(agent=nutrition_agent)

# Step 4: Test it
response = await nutrition_runner.run_debug("Calculate nutrition for 200g chicken breast", verbose=True)
```

### Tips:
- Look back at the recipe agent for patterns
- Use clear docstrings - the agent reads them!
- Test with verbose=True to debug
- Start simple, then enhance

**Good luck! 🚀**

In [23]:
# YOUR CODE HERE - Build the nutrition calculator tool

In [24]:
# Challenge: Nutrition Calculator (approximate DEMO values per 100 g)
NUTRITION_PER_100G = {
    "rice": {"calories": 130, "protein": 2.7, "carbs": 28.2, "fat": 0.3},  # cooked
    "chicken breast": {"calories": 165, "protein": 31.0, "carbs": 0.0, "fat": 3.6},  # cooked
    "broccoli": {"calories": 35, "protein": 2.4, "carbs": 7.2, "fat": 0.4},
    "apple": {"calories": 52, "protein": 0.3, "carbs": 13.8, "fat": 0.2},
    "salmon": {"calories": 208, "protein": 20.0, "carbs": 0.0, "fat": 13.0},  # cooked
}

def calculate_nutrition(food_item: str, serving_size: float = 100.0) -> dict:
    """Calculate approximate macro nutrients for a food weight in grams.

    Args:
        food_item: Food name: rice, chicken breast, broccoli, apple, or salmon.
        serving_size: Weight in grams, e.g. 200 means 200 grams.

    Returns:
        Dictionary with food, grams, calories, protein_g, carbs_g, and fat_g.
    """
    name = food_item.strip().lower()
    if name not in NUTRITION_PER_100G:
        return {"error": f"Unknown food '{food_item}'", "available": list(NUTRITION_PER_100G)}
    if serving_size <= 0:
        return {"error": "Serving size must be positive grams."}
    scale = serving_size / 100.0
    row = NUTRITION_PER_100G[name]
    return {"food": name, "serving_size_g": serving_size,
            "calories": round(row["calories"] * scale, 1),
            "protein_g": round(row["protein"] * scale, 1),
            "carbs_g": round(row["carbs"] * scale, 1),
            "fat_g": round(row["fat"] * scale, 1)}

# Direct tool tests before the model call.
assert calculate_nutrition("chicken breast", 200)["calories"] == 330
print("Direct tool test:", calculate_nutrition("broccoli", 150))

nutrition_agent = LlmAgent(
    name="nutrition_calculator_agent",
    model=azure_model,
    instruction=(
        "You are a nutrition assistant. Call calculate_nutrition separately for each food. "
        "The serving_size argument is weight in GRAMS. For meals, sum the returned calories, "
        "protein_g, carbs_g, and fat_g; show all totals clearly. "
        "Values are approximate estimates, not medical guidance. Do not invent unknown foods."
    ),
    tools=[FunctionTool(func=calculate_nutrition)],
)
nutrition_runner = InMemoryRunner(agent=nutrition_agent)
print("Nutrition agent created.")


Direct tool test: {'food': 'broccoli', 'serving_size_g': 150, 'calories': 52.5, 'protein_g': 3.6, 'carbs_g': 10.8, 'fat_g': 0.6}
Nutrition agent created.


In [25]:
# Challenge: run both nutrition tests with verbose tool-call output
async def show_nutrition_debug(question: str):
    print("\nQUESTION:", question)
    events = await nutrition_runner.run_debug(question, verbose=True)
    text_responses, tool_calls, tool_responses = analyze_events(events)
    print("TOOL CALLS:", tool_calls)
    print("TOOL RESPONSES:", tool_responses)
    for item in text_responses:
        print("ANSWER:", item)
    return events

single_food_events = await show_nutrition_debug("Calculate nutrition for 200g of chicken breast.")
meal_events = await show_nutrition_debug(
    "Calculate total nutrition for a meal of 200g chicken breast, 150g cooked rice, and 100g broccoli. "
    "Call the tool once per food and add the results."
)
# Reference totals from the demo data: 560 kcal; 68.45g protein; 49.5g carbs; 8.05g fat



QUESTION: Calculate nutrition for 200g of chicken breast.
nutrition_calculator_agent > The user wants me to calculate nutrition for 200g of chicken breast. I need to use the calculate_nutrition function with:
- food_item: "chicken breast" (as specified by the user)
- serving_size: 200 (in grams, as requested)

I'll make a single function call since there's only one food item mentioned.
nutrition_calculator_agent > [Calling tool: calculate_nutrition({'food_item': 'chicken breast', 'serving_size': 20...)]
nutrition_calculator_agent > [Tool result: {'food': 'chicken breast', 'serving_size_g': 200, 'calories': 330.0, 'protein_g': 62.0, 'carbs_g': 0...]
nutrition_calculator_agent > Great! I have the nutrition information for 200g of chicken breast. Let me present this clearly to the user with all the breakdowns and totals.Here's the nutritional information for **200g of chicken breast**:

| Nutrient | Amount |
|----------|--------|
| **Calories** | 330 kcal |
| Protein | 62 g |
| Carbohydr

---
## 🎓 What You've Learned

Congratulations! You've mastered the fundamentals of Google ADK:

✅ **Creating Agents** - Built agents with custom tools using `LlmAgent`

✅ **Custom Tools** - Wrapped Python functions with `FunctionTool`

✅ **Bring Your Own Model** - Powered ADK agents with Azure OpenAI through `LiteLlm`

✅ **MCP Integration** - Connected to external services with `McpToolset`

✅ **Observability** - Debugged with verbose mode and event inspection

✅ **Production Patterns** - Learned error handling, retry logic, and logging

### 📚 Resources:

- [Google ADK Documentation](https://adk.dev/)
- [ADK LiteLLM Model Connector](https://adk.dev/agents/models/litellm/)
- [Azure OpenAI v1 API](https://learn.microsoft.com/azure/ai-foundry/openai/api-version-lifecycle)
- [MCP Protocol Specification](https://modelcontextprotocol.io/)

**Happy Building! 🎉**